# Peak Probability Presentation (P3)

To generate **P3**, the first step is processing the spectra and converting the data from the **time domain (FID)** into the **frequency domain (spectrum)**. This transformation can be performed using different software packages such as **TopSpin** or **qMDD**.

Here, in order to work on this demo, please download the spectra processed with **qMDD** from [https://doi.org/10.5878/b8n4-x341](https://doi.org/10.5878/b8n4-x341) and place the **Spectra** folder in the `MRAi_P3-main` folder.
In this work, all processing steps were carried out on **NMRBox** in order to have access to **NMRPipe** and **NMRDraw**, but the same workflow can also be performed on a local computer using equivalent steps.

The `Spectra/` folder contains:  
- Four 2D uniformly sampled H–N HSQC spectra of different proteins (Ubiquitin, Azorin, Malt, and Tau).  
- Six additional spectra from different experiments on the Calmodulin and Malt proteins, recorded in NUS and processed with CS:  
  - HNCO  
  - HNCA  
  - HNcoCA  
  - HNcaCO  
  - CBCAcoNH  
  - CBCANH

Before proceeding, it is necessary to **import the required libraries** and **define the essential functions** that will be used in the analysis.  


In [ ]:
import numpy as np
import nmrglue as ng
import os, subprocess
import MRAi_P3 as P3
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':25})

def subp(axs,dic,data, tit,contour,Mode = True):
    N = 10; C = 1.3; cmap = [plt.cm.Greys_r, plt.cm.Reds_r]   # [+,-] contour maps
    cl = contour * C ** np.arange(N)
    uc_x = ng.pipe.make_uc(dic, data, dim=1)
    ppm_x = uc_x.ppm_scale()
    ppm_x_0, ppm_x_1 = uc_x.ppm_limits()
    uc_y = ng.pipe.make_uc(dic, data, dim=0)
    ppm_y = uc_y.ppm_scale()
    ppm_y_0, ppm_y_1 = uc_y.ppm_limits()

    axs.contour(data, cl, cmap=cmap[0],extent=(ppm_x_0, ppm_x_1, ppm_y_0, ppm_y_1))
    if Mode == True: axs.contour(-data, cl, cmap=cmap[1],extent=(ppm_x_0, ppm_x_1, ppm_y_0, ppm_y_1))

    # decorate the axes
    axs.set_ylabel(dic["FDF1LABEL"]+" (ppm)")
    axs.set_xlabel(dic["FDF2LABEL"]+" (ppm)")
    axs.set_title(tit,y=1,pad=-25, loc='left')
    axs.locator_params(axis="x", nbins=4)
    axs.locator_params(axis="y", nbins=4)
    axs.invert_xaxis()
    axs.invert_yaxis()
    
def PLOT_2D(S_dirc,S_contour  = 0.005, P_contour  = 0.1):
    dic, S = ng.pipe.read(S_dirc)
    P_dirc = os.path.splitext(S_dirc)[0] + '.P3'
    dic, P = ng.pipe.read(P_dirc)
    
    # Plot side by side
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))

    subp(ax[0], dic, S/np.max(S), "Spectrum", S_contour)
    subp(ax[1], dic, P, "P3", P_contour)

    # Hide y-axis for the second panel
    ax[1].axes.yaxis.set_visible(False)

    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    plt.show()

def SKY(US,axis):
    MAX = np.max(US,axis=axis); MIN = np.min(US,axis=axis)
    return np.where(np.abs(MAX)>=np.abs(MIN),MAX,MIN)

def PLOT_3D(S_dirc,S_contour  = 0.005, P_contour  = 0.1, axis = 0):
    Dic, S = ng.pipe.read(S_dirc)
    if axis == 0: dirc = os.path.splitext(S_dirc)[0] + '_' + Dic["FDF2LABEL"] + '.' + Dic["FDF1LABEL"] + '.ft2'
    if axis == 1: dirc = os.path.splitext(S_dirc)[0] + '_' + Dic["FDF2LABEL"] + '.' + Dic["FDF3LABEL"] + '.ft2'
    if axis == 2: dirc = os.path.splitext(S_dirc)[0] + '_' + Dic["FDF1LABEL"] + '.' + Dic["FDF3LABEL"] + '.ft2'
    PLOT_2D(dirc,S_contour,P_contour)

# 2D Spectra

For **2D spectra**, the first step is to **read the processed data**.  
We use the **nmrglue** library for handling **NMRPipe** files, and **nmrDraw** to visualize the processed spectrum.

First, we specify the path to the processed spectrum and then, read the same spectrum in **Pipe format** with **nmrglue** for further processing:    

In [ ]:
S_dirc = "Spectra/2D/Ubiquitin.ft2"
dic, S = ng.pipe.read(S_dirc)

### P3

For the **2D case**, generating **P3** is straightforward.  
It requires specifying the **directory of the trained network** and defining the **intensity threshold** (typically `0.5`, which is the default and usually does not need adjustment).  

Once calculated, the **P3 spectrum** is saved in the same location as the original data in **`.P3` format**, which can then be visualized directly with *nmrDraw*.  


In [ ]:
Dirc_2D = "Trained/Probability_2D/US/"
Spe_The = 0.5

# Generate P3 from the spectrum using the trained model
P = P3.StoP3.P3_2D(S, Dirc_2D,Spe_The=Spe_The)

# Save P3 in the same directory as the original spectrum
P_dirc = os.path.splitext(S_dirc)[0] + '.P3'
ng.pipe.write(P_dirc,dic,np.float32(P),overwrite='True')

#### Comparing Spectrum and P3

Finally, we can visualize the **original spectrum** and the corresponding **P3 map** side by side for direct comparison.  
Here, different contour levels are applied for clarity.  


In [ ]:
S_dirc = "Spectra/2D/Ubiquitin.ft2"
PLOT_2D(S_dirc,S_contour  = 0.005, P_contour  = 0.1)

### Peak Picking with P3 (P5)

The **P3 map** can be used to automatically generate reliable peak lists.  
The key parameters are:

- **Threshold (`PL_The`)**: the minimum probability above which a signal is considered a peak  
  *(default = 0.2, i.e. 20%)*.  
- **Peak type (`PN_Peak`)**: whether to include only positive peaks (`'P'`) or both positive and negative peaks (`'PN'`).  
- **Format**: determines the output format *(here, `'Pipe'` is used)*.  

By default, peaks above **20% probability with positive intensity** are included.  

In [ ]:
Spec_type  = 'HSQC'   # spectrum type (adjust if needed)            
PL_The     = 0.2        # threshold for peak selection
PN_Peak    = 'P'        # 'P' = positive only, 'PN' = positive + negative
Format='Pipe'
# Save P3 peak list
P5_dirc = os.path.splitext(S_dirc)[0] + '.P5'
P3.P3toP5.main(P_dirc,S_dirc,PN_Peak,PL_The,Format,P5_dirc)

The peak list is initially saved as a **`.P5` file**, which is then converted to:  
- **`.peaks`** (CYANA format)  
- **`.tab`** (Pipe format)

**NOTE:** `NMRPIPE` is required to convert `.P5` files to other formats.

All files are stored in the same directory as the input spectrum.  


In [ ]:
# Convert to CYANA peak list
Peaks_dirc = os.path.splitext(S_dirc)[0] + '.peaks'
P3.P5toPeaks.main(P5_dirc,S_dirc,Spec_type,Peaks_dirc)

# Convert to Pipe tab format
Tab_dirc = os.path.splitext(S_dirc)[0] + '.tab'
P3.PeakstoTab.main(Peaks_dirc,S_dirc,Tab_dirc)

##### Visualizing the Peak List in nmrDraw

To view the generated peak list in **nmrDraw**:

1. Open the processed spectrum in *nmrDraw*.  
2. Right-click on **Peak** and select **Peak Detection**.  
3. In the dialog, choose **Peak Table** and select the corresponding `.tab` file (e.g., `Ubiquitin.tab`).  
4. Load the table; the peaks will be displayed on top of the spectrum.  


## Running the Full Workflow from Terminal  

All of the above steps can also be executed directly from the **terminal**.  

Open a terminal in the `MRAi_P3-main/` directory and activate your **Conda environment**:  

```bash
conda activate mrai_p3
```

In order to fix the parameters, it is necessary to run:
```bash
python Input_P3.py -spec Spectra/2D/Ubiquitin.ft2 -dim 2 -o Inputs/Ubiquitin.input
```
where `-spec` specifies the spectrum to be processed,`-dim` specifies the dimensionality of the spectrum, and `-o`specifies the location for saving the parameters. Here, `Ubiquitin.input` contains the processing and analysis parameters that were defined in the workflow above for Ubiquitin.

After fixing the parameters, run:
```bash
python P3_Master.py Inputs/Ubiquitin.input
```

**Note:** P3 for all 2D datasets:  

```bash
python P3_Master.py Inputs/Ubiquitin.input Inputs/Azurin.input Inputs/Malt.input Inputs/Tau.input
```

In [ ]:
S_dirc = "Spectra/2D/Ubiquitin.ft2"
PLOT_2D(S_dirc,S_contour  = 0.005, P_contour  = 0.1)
S_dirc = "Spectra/2D/Azurin.ft2"
PLOT_2D(S_dirc,S_contour = 0.005, P_contour = 0.1)
S_dirc = "Spectra/2D/Malt.ft2"
PLOT_2D(S_dirc,S_contour = 0.01, P_contour = 0.1)
S_dirc = "Spectra/2D/Tau.ft2"
PLOT_2D(S_dirc,S_contour  = 0.004, P_contour = 0.1)

---
# 3D Spectra  

The **3D case** is very similar to the 2D workflow.  

As before, the first step is to **read the processed data** using the **nmrglue** library for handling **NMRPipe** files, and to use **nmrDraw** for visualization.  
The spectrum can then be read in **Pipe format** with *nmrglue* for further analysis and processing.  

##### For improved visualization, we also generate **2D projections** along all three dimensions of the 3D spectrum.  


In [ ]:
S_dirc = "Spectra/3D_CaM/HNCA.ft3"

# Generate 2D projections from the 3D spectrum
subprocess.run(["./3Dto2Dproj.sh", S_dirc])

# Read 3D spectum
dic, S = ng.pipe.read(S_dirc)

### P3 without Support  

For the **3D case**, generating **P3** is similar to the 2D workflow.  
It requires specifying the **directories of the trained networks** and defining two key parameters:  

- **`Fil_The`**: Threshold for the region of interest (ROI).  
  *Default = `0.025**3`, which usually does not need adjustment.*  

- **`PN_The`**: Controls whether only positive peaks are considered (`0`), or both positive and negative peaks are included  
  *(e.g., `0.01` for 3D data, to avoid cancellation of peaks in the projections).*  

Once calculated, the **P3 spectrum** is saved in the same directory as the original data in **`.P3` format**, which can be visualized directly with *nmrDraw*.  
In addition, **2D projections** are generated from the 3D P3 for easier inspection.  


In [ ]:
Dirc_2D = "Trained/Probability_2D/PR"
Dirc_3D = "Trained/Probability_3D"
Fil_The = 0.025**3
PN_The = 0

# Generate P3 from the spectrum using the trained model
P = P3.StoP3.P3_3D(S,Dirc_3D,Dirc_2D,Filter_The=Fil_The,PN_The=PN_The) 

# Save P3 in the same directory as the original spectrum
P_dirc = os.path.splitext(S_dirc)[0] + '.P3'
ng.pipe.write(P_dirc,dic,np.float32(P),overwrite='True')

# Generate 2D projections from the 3D P3
subprocess.run(["./3Dto2Dproj.sh", P_dirc])

In [ ]:
PLOT_3D(S_dirc,S_contour  = 0.01, P_contour  = 0.1,axis = 1)

# P3 with Support  

P3 also has the potential to use **supporting spectra** to improve the results through **co-processing**.  

In this example, for the same 3D spectrum we use the **HNCO H–N projection** as additional support to guide the P3 calculation with:  

- **`Support`** → the supporting spectrum loaded via `ng.pipe.read()`.  
- **`Support_axis`** → the axis of projection used from the support spectrum  
  *(0 = Z, 1 = Y, 2 = X). For 2D HSQC, use `1`.*  
- **`Support_The`** → threshold applied to the support spectrum to define a mask for the ROI.

**Note:** `Fil_The` should be `0.025**2` becouse of support.


In [ ]:
Dirc_2D = "Trained/Probability_2D/PR"
Dirc_3D = "Trained/Probability_3D"
Fil_The = 0.025**2
PN_The = 0

# Load support spectrum
Support_dirc = "Spectra/3D_CaM/HNCO.ft3"
_ , Support = ng.pipe.read(Support_dirc)

# projected axis in the 3D support spectrum  0,1,2 - Z, Y, X (N,C,H)
# For 2D HSQC use 1
Support_axis = 1
# Thereshould in the P3 of the support spectrum. THis is used as a mask for for RoI 
Support_The = 0.25

# Generate P3 from the spectrum using the trained model
P = P3.StoP3.P3_3D(S,Dirc_3D,Dirc_2D,Support=Support,Support_axis=Support_axis,Support_The=Support_The,Filter_The=Fil_The,PN_The=PN_The) 

# Save P3 in the same directory as the original spectrum
P_dirc = os.path.splitext(S_dirc)[0] + '.P3'
ng.pipe.write(P_dirc,dic,np.float32(P),overwrite='True')

# Generate 2D projections from the 3D P3
subprocess.run(["./3Dto2Dproj.sh", P_dirc])

In [ ]:
PLOT_3D(S_dirc,S_contour  = 0.01, P_contour  = 0.1,axis = 1)

### Peak Picking with P3

In [ ]:
Spec_type  = 'defult'   # spectrum type (adjust if needed)            
PL_The     = 0.1        # threshold for peak selection
PN_Peak    = 'P'        # 'P' = positive only, 'PN' = positive + negative
Format     = 'Pipe'     # output format


# Save P3 peak list
P5_dirc = os.path.splitext(S_dirc)[0] + '.P5'
P3.P3toP5.main(P_dirc,S_dirc,PN_Peak,PL_The,Format,P5_dirc)

# Convert to CYANA peak list
Peaks_dirc = os.path.splitext(S_dirc)[0] + '.peaks'
P3.P5toPeaks.main(P5_dirc,S_dirc,Spec_type,Peaks_dirc)

# Convert to Pipe tab format
Tab_dirc = os.path.splitext(S_dirc)[0] + '.tab'
P3.PeakstoTab.main(Peaks_dirc,S_dirc,Tab_dirc)

# Convert 3D to 2D Pipe tab format
P3.Tab3Dto2Dproj.main(Tab_dirc)


## Running the Full Workflow from Terminal  

All of the above steps can also be executed directly from the **terminal**.  

Open a terminal in the `MRAi_P3-main/` directory, activate your **Conda environment**, and run:  

```bash
python P3_Master.py Inputs/HNCA_CaM.input
```
Here, `HNCA_CaM.input` contains the processing and analysis parameters defined above for the **HNCA** experiment.

**Note**: If no support is used, set *Support_dirc = **NO***

---

We can generate **P3** for all other 3D spectra using **HNCO Support**.  

1. Create the following input files by using `CaM.input` and editing parameters as needed:  
   - `Inputs/HNCO_CaM.input`  
   - `Inputs/HNcoCA_CaM.input`  
   - `Inputs/HNcaCO_CaM.input`  
   - `Inputs/CBCANH_CaM.input`  
   - `Inputs/CBCAcoNH_CaM.input`  

**Note:** For `HNCO_CaM.input`, set `Support_dirc = NO` since the support and spectrum are the same.  

2. Run **P3** for all 3D datasets:  

```bash
python P3_Master.py Inputs/HNCO_CaM.input Inputs/HNCA_CaM.input Inputs/HNcoCA_CaM.input Inputs/HNcaCO_CaM.input Inputs/CBCANH_CaM.input Inputs/CBCAcoNH_CaM.input
```


### 3D Malt
For Malt protein:
```bash
python P3_Master.py Inputs/HNCA_Malt.input Inputs/HNcoCA_Malt.input Inputs/HNcaCO_Malt.input Inputs/CBCANH_Malt.input 
```